# Módulo 3: Agentes de IA
- **Sesión:** 01 — Fundamentos de agentes de IA
- **Tema:** Patrones agénticos — workflows frente a agentes
- **Fecha:** 20/10/2026

---
## Fundamentos — Sistemas agénticos: workflow y agente

Un **agente** es un sistema que percibe su entorno y actúa sobre él (Russell y Norvig, 2021). En las aplicaciones con modelos de lenguaje, el bloque básico es el **LLM aumentado**: un modelo con acceso a herramientas, a recuperación de información y a memoria (Schluntz y Zhang, 2024). Con ese bloque se construyen dos tipos de sistemas, que se distinguen por **quién decide el siguiente paso**:

| | Workflow | Agente |
|---|---|---|
| Quién decide el flujo | El desarrollador, en el código | El modelo, en tiempo de ejecución |
| Función del modelo | Participa en pasos fijos: extrae, clasifica, redacta | Elige las herramientas, su orden y cuándo terminar |



## 1. — Configuración: modelo, datos y herramientas


In [14]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

In [15]:

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



#### Definiciones de herramientas

In [28]:
from langchain.tools import tool

# Datos simulados de TiendaYa
PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "77100": "entregado el 10/10/2026 en Lima",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}

POLITICAS = {
    "devoluciones": "15 días calendario desde la entrega; 7 días para productos electrónicos.",
    "envios": "Envío gratuito desde S/ 150; envío express en Lima por S/ 24.90.",
}


@tool
def consultar_pedido(numero_pedido: str) -> str:
    """
        Consulta el estado de un pedido de TiendaYa. 
        Usar cuando el cliente menciona un número de pedido.
    """
    return PEDIDOS.get(numero_pedido, f"No existe el pedido {numero_pedido}.")


@tool
def consultar_politica(tema: str) -> str:
    """
        Devuelve la política de TiendaYa sobre un tema: 'devoluciones' o 'envios'.
    """
    return POLITICAS.get(tema.lower(), "No existe una política sobre ese tema.")


print(consultar_pedido.invoke(
                                {
                                    "numero_pedido": "48213"
                                }
                            ))

print(consultar_politica.invoke({"tema": "envios"}))

#print(consultar_pedido(numero_pedido ="48213"))
#print(consultar_politica(tema="envios"))



en ruta, llega el 26/10/2026 a Arequipa
Envío gratuito desde S/ 150; envío express en Lima por S/ 24.90.


## 2 — Workflow de encadenamiento -> *prompt chaining*


![Workflow de encadenamiento](graficos/M3_S01_patrones_paso1_encadenamiento.png)


In [30]:
CONSULTA = "Hola, ¿dónde está mi pedido 48213?"

# Paso 1: el modelo extrae el número de pedido
numero = llm.invoke(
    f"""
       Extrae solo el número de pedido de este mensaje, sin otro texto: {CONSULTA}
     """
).text.strip()
print("1. Número extraído:", numero)

# Paso 2: el código consulta el pedido (siempre esta herramienta)
estado = consultar_pedido.invoke({"numero_pedido": numero})
print("2. Estado del pedido:", estado)

# Paso 3: el modelo redacta la respuesta, anclada al dato
prompt_respuesta =  f"""
                        Responde de forma breve y cordial al cliente usando SOLO el dato indicado.
                        Si la consulta pide algo que no está en el dato, indica que no dispones de esa información.
                    
                        Consulta: {CONSULTA}
                    
                        Dato: {estado}
                    """
print("3. Prompt para la respuesta :", prompt_respuesta)
respuesta = llm.invoke(prompt_respuesta).text

print(respuesta)

1. Número extraído: 48213
2. Estado del pedido: en ruta, llega el 26/10/2026 a Arequipa
3. Prompt para la respuesta : 
                        Responde de forma breve y cordial al cliente usando SOLO el dato indicado.
                        Si la consulta pide algo que no está en el dato, indica que no dispones de esa información.

                        Consulta: Hola, ¿dónde está mi pedido 48213?

                        Dato: en ruta, llega el 26/10/2026 a Arequipa
                    
Su pedido 48213 está en ruta y llegará el 26/10/2026 a Arequipa.


### Ejercicios

2.1. Agregar un nuevo pedido "60001" e indicar en su estado "Entregado en Tecsup el 22/10/2026 para el Fashion Tec"

2.2. Crear un procedimiento llamado workflow_pedido donde se pase la CONSULTA y me devuelva la respuesta del sistema.

2.3. Realizar las siguientes consultas usando la solución del paso 2.2

      - "Buenos días, ¿dónde está mi pedido 60001?"

      - "Buenos tardes, ¿cuál es el estado de mi pedido 77100?"

      - "Buenos noches, ¿dame la situación de mi pedido 50001?"

      

In [35]:

def workflow_pedido(consulta : str ) -> str :
    """
    
    """

    # Paso 1: el modelo extrae el número de pedido
    numero = llm.invoke(
        f"""
        Extrae solo el número de pedido de este mensaje, sin otro texto: {consulta}
        """
    ).text.strip()
    print("1. Número extraído:", numero)

    # Paso 2: el código consulta el pedido (siempre esta herramienta)
    estado = consultar_pedido.invoke({"numero_pedido": numero})
    print("2. Estado del pedido:", estado)

    # Paso 3: el modelo redacta la respuesta, anclada al dato
    prompt_respuesta =  f"""
                            Responde de forma breve y cordial al cliente usando SOLO el dato indicado.
                            Si la consulta pide algo que no está en el dato, indica que no dispones de esa información.
                        
                            Consulta: {consulta}
                        
                            Dato: {estado}
                        """
    print("3. Prompt para la respuesta :", prompt_respuesta)
    respuesta = llm.invoke(prompt_respuesta).text

    return respuesta

# Uso
CONSULTA = "Hola, ¿dónde está mi pedido 48213?"
print(workflow_pedido(CONSULTA))

1. Número extraído: 48213
2. Estado del pedido: en ruta, llega el 26/10/2026 a Arequipa
3. Prompt para la respuesta : 
                            Responde de forma breve y cordial al cliente usando SOLO el dato indicado.
                            Si la consulta pide algo que no está en el dato, indica que no dispones de esa información.

                            Consulta: Hola, ¿dónde está mi pedido 48213?

                            Dato: en ruta, llega el 26/10/2026 a Arequipa
                        
Su pedido está en ruta y llegará el 26/10/2026 a Arequipa.


## 3 — Workflow de enrutamiento -> routing


![Workflow de enrutamiento](graficos/M3_S01_patrones_paso2_enrutamiento.png)


#### Versión 1

In [33]:

#CONSULTA = "Hola, ¿dónde está mi pedido 48213?"
CONSULTA = "¿Cuántos días tengo para devolver un celular?"
#CONSULTA = "¿Tienen tienda física en Trujillo?"

# Paso 1: clasificar
categoria = llm.invoke(
    f"""
        Clasifica la consulta del cliente en una sola palabra: PEDIDO, POLITICA u OTRO.

        Consulta: {CONSULTA}
    """
).text.strip().upper()

# Paso 2 : enrutar
if "PEDIDO" in categoria:
    # Paso 1: el modelo extrae el número de pedido
    numero = llm.invoke(
        f"""
        Extrae solo el número de pedido de este mensaje, sin otro texto: {CONSULTA}
        """
    ).text.strip()
    print("1. Número extraído:", numero)

    # Paso 2: el código consulta el pedido (siempre esta herramienta)
    estado = consultar_pedido.invoke({"numero_pedido": numero})
    print("2. Estado del pedido:", estado)

    # Paso 3: el modelo redacta la respuesta, anclada al dato
    respuesta = llm.invoke(
        f"""
            Responde de forma breve y cordial al cliente usando SOLO el dato indicado.
            Si la consulta pide algo que no está en el dato, indica que no dispones de esa información.
        
            Consulta: {CONSULTA}
        
            Dato: {estado}
        """
    ).text

if "POLITICA" in categoria:

    tema = llm.invoke(
        f"""
            ¿La consulta trata de 'devoluciones' o de 'envios'?.
            Responde solo con esa palabra.
             
            Consulta: {CONSULTA}
        """
    ).text.strip().lower()
    
    politica = consultar_politica.invoke({"tema": tema})
    
    respuesta = llm.invoke(
        f"""
            Responde de forma breve al cliente. 
            
            Consulta: {CONSULTA}
            
            Política: {politica}
        """
    ).text
else:
    respuesta = "Su consulta será atendida por un operador."

print(respuesta)

Tienes 7 días calendario desde que recibas el celular para devolverlo.


#### Versión 2

In [ ]:

CONSULTA = "Hola, ¿dónde está mi pedido 48213?"
#CONSULTA = "¿Cuántos días tengo para devolver un celular?"
#CONSULTA = "¿Tienen tienda física en Trujillo?"

# Paso 1: clasificar
categoria = llm.invoke(
    f"""
        Clasifica la consulta del cliente en una sola palabra: PEDIDO, POLITICA u OTRO.

        Consulta: {CONSULTA}
    """
).text.strip().upper()

# Paso 2 : enrutar
if "PEDIDO" in categoria:
    respuesta = workflow_pedido(CONSULTA)
if "POLITICA" in categoria:

    tema = llm.invoke(
        f"""
            ¿La consulta trata de 'devoluciones' o de 'envios'?.
            Responde solo con esa palabra.
             
            Consulta: {CONSULTA}
        """
    ).text.strip().lower()
    
    politica = consultar_politica.invoke({"tema": tema})
    
    respuesta = llm.invoke(
        f"""
            Responde de forma breve al cliente. 
            
            Consulta: {CONSULTA}
            
            Política: {politica}
        """
    ).text
else:
    respuesta = "Su consulta será atendida por un operador."

print(respuesta)

1. Número extraído: 48213
2. Estado del pedido: en ruta, llega el 26/10/2026 a Arequipa
3. Prompt para la respuesta : 
                            Responde de forma breve y cordial al cliente usando SOLO el dato indicado.
                            Si la consulta pide algo que no está en el dato, indica que no dispones de esa información.

                            Consulta: Hola, ¿dónde está mi pedido 48213?

                            Dato: en ruta, llega el 26/10/2026 a Arequipa
                        
Su consulta será atendida por un operador.


### Ejercicios de Workflow Routing

3.1. Agregar un nueva politica de "cambios" e indicar en su estado "Cambios de talla o color: 30 días desde la entrega, con la etiqueta puesta."

3.2. Crear un procedimiento llamado workflow_politica donde se pase la CONSULTA y me devuelva la respuesta de la politica.

3.3. Realizar las siguientes consultas usando la solución del paso 3.2

      - "Buenos días, ¿dónde está mi pedido 48213?"

      - "Buenos tardes, ¿Cuántos días tengo para devolver un celular?"

3.4. Agregar la categoria 'RECLAMO' que me responda con un texto fijo "Su reclamo fue registrado. Un supervisor lo contactará en 24 horas."

## 4 — Límite del workflow

![Límite del workflow](graficos/M3_S01_patrones_paso3_limite.png)


## 5 — Agente: el modelo decide el flujo


![Agente: ciclo ReAct](graficos/M3_S01_patrones_paso4_agente.png)
